# 10 — Batch Normalization: a training-time story

A downstream layer never sees the dataset directly. It sees values made by the layers before it. As those earlier layers learn, the distribution of those values can move—even when we hold one minibatch fixed.

We will first watch that drift across many simulated upstream updates. We will then see why an activation cares about the drift, and finally follow the same effect through a stack. BatchNorm is the intervention that on every training batch recentres and rescales each feature before the activation.

For one feature $x_j$ across a batch of $m$ examples, BatchNorm calculates

$$\mu_j = \frac{1}{m} \sum_i x_{ij}, \qquad \sigma_j^2 = \frac{1}{m} \sum_i (x_{ij} - \mu_j)^2, \qquad \hat{x}_{ij} = (x_{ij} - \mu_j) / \sqrt{\sigma_j^2 + \varepsilon}, \qquad y_{ij} = \gamma_j \hat{x}_{ij} + \beta_j,$$

> Run this with the notebook extra installed: `pip install -e ".[notebooks]"`

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from numpy.typing import ArrayLike, NDArray

from bonsaigrad import Leaf
from bonsaigrad.nn import CrossEntropyLoss, Linear, MSELoss, Module, ReLU, Sequential, Tanh
from bonsaigrad.nn.norm import BatchNorm1d

rng = np.random.default_rng(7)
plt.style.use("seaborn-v0_8-whitegrid")

COLORS = {"before": "#c1543c", "after": "#2878b5"}


def plot_feature_histograms(axes, before, after, title):
    bins = np.linspace(min(before.min(), after.min()), max(before.max(), after.max()), 35)
    for ax, values, color, label in zip(
            axes,
            (before, after),
            (COLORS["before"], COLORS["after"]),
            ("before", "after BatchNorm"),
    ):
        ax.hist(values.ravel(), bins=bins, color=color, alpha=0.8)
        ax.axvline(0, color="black", linewidth=1)
        ax.set_title(f"{label}: mean {values.mean():+.2f}, std {values.std():.2f}")
        ax.set_xlabel(title)
        ax.spines[["top", "right"]].set_visible(False)


## 1. Start with one fixed minibatch

We keep one minibatch of 320 examples fixed. A `Linear(2, 2)` layer maps each example to two outputs, $z = XW + b$, which become the next layer's input. The input never changes, but the layer's weights do—so its output values change too.

The scatter plots show outputs at updates 0, 12, and 24. The top row shows the raw values moving. The bottom row applies BatchNorm to the same values, centring each feature near 0 and scaling it near 1. The line plots track those centres and spreads across training.

This changing distribution of values inside a network is what the original BatchNorm paper called **internal covariate shift**.

In [ ]:
# 1. Make one fixed minibatch and its fixed regression targets.
inputs = rng.normal(size=(320, 2))
target_weights = np.array([[2.24, -0.14], [-0.56, 0.23]])
target_bias = np.array([2.3, -1.6])
target_outputs = inputs @ target_weights + target_bias

# 2. Start the upstream layer away from those targets.
upstream = Linear(2, 2, rng)
upstream.weight.data[:] = [[0.8, -0.3], [-0.2, 0.5]]
upstream.bias.data[:] = [0.0, 0.0]
batch_norm = BatchNorm1d(2)
mse_loss = MSELoss()
updates = 24
learning_rate = 0.2

# 3. Train the upstream layer, watching what it passes downstream.
batch = Leaf(inputs)
raw_means, raw_stds = [], []
normalized_means, normalized_stds = [], []
raw_tanh_slopes, normalized_tanh_slopes = [], []
losses = []
snapshots = {}
snapshot_updates = (0, updates // 2, updates)

for update in range(updates + 1):
    prediction = upstream(batch)
    raw_outputs = prediction.data
    normalized_outputs = batch_norm(Leaf(raw_outputs)).data

    raw_means.append(raw_outputs.mean(axis=0))
    raw_stds.append(raw_outputs.std(axis=0))
    normalized_means.append(normalized_outputs.mean(axis=0))
    normalized_stds.append(normalized_outputs.std(axis=0))
    raw_tanh_slopes.append((1 - np.tanh(raw_outputs) ** 2).mean())
    normalized_tanh_slopes.append((1 - np.tanh(normalized_outputs) ** 2).mean())

    if update in snapshot_updates:
        snapshots[update] = raw_outputs, normalized_outputs

    if update < updates:
        loss = mse_loss(prediction, target_outputs)
        losses.append(loss.data.item())
        loss.wire()
        upstream.weight.data -= learning_rate * upstream.weight.grad
        upstream.bias.data -= learning_rate * upstream.bias.grad
        loss.rest()

raw_means = np.array(raw_means)
raw_stds = np.array(raw_stds)
normalized_means = np.array(normalized_means)
normalized_stds = np.array(normalized_stds)
final_loss = mse_loss(upstream(batch), target_outputs).data.item()

# 4. Compare the same clouds before and after BatchNorm.
fig, axes = plt.subplots(2, 3, figsize=(12, 7), sharex=True, sharey=True)
for column, update in enumerate(snapshot_updates):
    raw_outputs, normalized_outputs = snapshots[update]
    for row, (values, label, color) in enumerate((
            (raw_outputs, "without BatchNorm", COLORS["before"]),
            (normalized_outputs, "with BatchNorm", COLORS["after"]),
    )):
        axes[row, column].scatter(values[:, 0], values[:, 1], s=12, alpha=0.45, color=color)
        axes[row, column].set_title(f"update {update}: {label}")
    for ax in axes[:, column]:
        ax.axhline(0, color="black", linewidth=0.7)
        ax.axvline(0, color="black", linewidth=0.7)
axes[0, 0].set_ylabel("feature 1")
axes[1, 0].set_ylabel("feature 1")
for ax in axes[1]:
    ax.set_xlabel("feature 0")
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 3, figsize=(14, 3.5))
for feature in range(2):
    axes[0].plot(raw_means[:, feature], label=f"feature {feature} without")
    axes[0].plot(normalized_means[:, feature], "--", color=COLORS["after"], alpha=0.7)
    axes[1].plot(raw_stds[:, feature], label=f"feature {feature} without")
    axes[1].plot(normalized_stds[:, feature], "--", color=COLORS["after"], alpha=0.7)
axes[2].plot(raw_tanh_slopes, color=COLORS["before"], label="without BatchNorm")
axes[2].plot(normalized_tanh_slopes, color=COLORS["after"], label="with BatchNorm")
for ax, title, ylabel in zip(
        axes,
        ("feature means", "feature standard deviations", "Tanh responsiveness"),
        ("mean", "standard deviation", "average local slope"),
):
    ax.set_title(title)
    ax.set_xlabel("upstream update")
    ax.set_ylabel(ylabel)
    ax.spines[["top", "right"]].set_visible(False)
axes[0].axhline(0, color="black", linewidth=0.7)
axes[1].axhline(1, color="black", linewidth=0.7)
axes[0].legend(fontsize=8)
axes[2].legend(fontsize=8)
plt.tight_layout()
plt.show()

print("The minibatch never changed; `wire()` and gradient descent changed the parameters.")
print(f"MSE: {losses[0]:.3f} before training -> {final_loss:.3f} after {updates} updates")
print("final raw mean and std:        ", np.round(raw_means[-1], 2), np.round(raw_stds[-1], 2))
print("final normalized mean and std: ", np.round(normalized_means[-1], 2), np.round(normalized_stds[-1], 2))

## 2. Why the activation function cares

A large positive shift is troublesome in different ways for two common activations:

- **Tanh** compresses large magnitudes near $-1$ or $+1$. Its local slope approaches zero there, so a gradient has little effect.
- **ReLU** is exactly zero for negative inputs. A large negative shift can leave many units inactive for this batch.

The responsiveness plot above used Tanh’s local slope, $1 - \tanh^2(z)$, as a concrete consequence of drift. We will now make both failure modes intentionally obvious, then apply the same per-feature batch normalization. The point is not that BatchNorm guarantees perfect gradients; it keeps many values in a region where the activation is responsive.

In [ ]:
def activation_input_gradients(values, activation):
    """Differentiate the sum of activation outputs through BonsaiGrad."""
    inputs = Leaf(values)
    activation(inputs).sum().wire()
    return inputs.grad


tanh_inputs = rng.normal(loc=3.4, scale=1.4, size=(500, 6))
relu_inputs = rng.normal(loc=-2.0, scale=1.5, size=(500, 6))
tanh_normalized = BatchNorm1d(6)(Leaf(tanh_inputs)).data
relu_normalized = BatchNorm1d(6)(Leaf(relu_inputs)).data

tanh = Tanh()
relu = ReLU()
tanh_gradients = activation_input_gradients(tanh_inputs, tanh)
tanh_normalized_gradients = activation_input_gradients(tanh_normalized, tanh)
relu_gradients = activation_input_gradients(relu_inputs, relu)
relu_normalized_gradients = activation_input_gradients(relu_normalized, relu)

fig, axes = plt.subplots(2, 2, figsize=(12, 7))
plot_feature_histograms(axes[0], tanh_inputs, tanh_normalized, "values entering Tanh")
plot_feature_histograms(axes[1], relu_inputs, relu_normalized, "values entering ReLU")
plt.tight_layout()
plt.show()

print("Tanh average local slope")
print(f"  shifted:    {tanh_gradients.mean():.3f}")
print(f"  normalized: {tanh_normalized_gradients.mean():.3f}")
print("\nReLU active fraction")
print(f"  shifted:    {relu_gradients.mean():.1%}")
print(f"  normalized: {relu_normalized_gradients.mean():.1%}")

## 3. Through a whole stack

One layer can look manageable while the effect compounds with depth. Here we use fixed random affine layers and deliberately biased/scaled weights. The plot tracks the values *just before each activation*. Without BatchNorm, each layer receives a different scale and offset. With BatchNorm, each layer begins from a standardized batch (before its learned $\gamma$ and $\beta$ are changed by training).

Switch `activation_name` between `"tanh"` and `"relu"`, then rerun this cell to compare the two nonlinearities.

In [ ]:
activation_name = "tanh"  # try "relu" too
activation = Tanh() if activation_name == "tanh" else ReLU()

stack_inputs = rng.normal(size=(256, 8))
layer_specs = [
    (rng.normal(scale=1.0, size=(8, 8)) * scale, np.full(8, bias))
    for scale, bias in ((1.7, 1.4), (1.3, -1.1), (1.8, 1.8), (1.5, -1.5))
]
linear_layers = [Linear(8, 8, rng) for _ in layer_specs]
batch_norm_layers = [BatchNorm1d(8) for _ in layer_specs]
for linear, (weight, bias) in zip(linear_layers, layer_specs):
    linear.weight.data[:] = weight
    linear.bias.data[:] = bias


def trace_stack(use_batch_norm):
    values = Leaf(stack_inputs)
    pre_activations = []
    for linear, batch_norm in zip(linear_layers, batch_norm_layers):
        values = linear(values)
        if use_batch_norm:
            values = batch_norm(values)
        pre_activations.append(values.data)
        values = activation(values)
    return pre_activations


without_batch_norm = trace_stack(use_batch_norm=False)
with_batch_norm = trace_stack(use_batch_norm=True)

fig, axes = plt.subplots(2, len(layer_specs), figsize=(16, 6), sharey=False)
for index, (raw, normalized_values) in enumerate(zip(without_batch_norm, with_batch_norm), start=1):
    for ax, values, title, color in (
            (axes[0, index - 1], raw, "without BatchNorm", COLORS["before"]),
            (axes[1, index - 1], normalized_values, "with BatchNorm", COLORS["after"]),
    ):
        ax.hist(values.ravel(), bins=30, color=color, alpha=0.8)
        ax.axvline(0, color="black", linewidth=0.8)
        ax.set_title(f"layer {index}: {title}")
        ax.set_xlabel("pre-activation value")
        ax.spines[["top", "right"]].set_visible(False)
axes[0, 0].set_ylabel("count")
axes[1, 0].set_ylabel("count")
plt.suptitle(f"Distribution seen by each {activation_name.title()} activation", y=1.02, fontsize=14)
plt.tight_layout()
plt.show()


def responsiveness(values):
    return activation_input_gradients(values, activation).mean()


label = "average Tanh slope" if activation_name == "tanh" else "ReLU active fraction"

print(label)
for index, (raw, normalized_values) in enumerate(zip(without_batch_norm, with_batch_norm), start=1):
    print(f"  layer {index}: {responsiveness(raw):.3f} without  |  {responsiveness(normalized_values):.3f} with")

## 4. Training with minibatches

Now train two matching networks to classify points from three interleaved spirals. Each hidden layer is `Linear → Tanh`; in one network, BatchNorm sits between `Linear` and `Tanh`. Both start with the same weights and see the same shuffled minibatches.

After every epoch, we measure cross-entropy and accuracy on a separate test set. BatchNorm switches to evaluation mode for that measurement, so it uses the running mean and variance collected during training.

This one small run is not a benchmark: its purpose is to make minibatch statistics and evaluation-time running statistics visible.

In [ ]:
epochs = 15
batch_size = 256
learning_rate = 0.05
hidden_sizes = (16, 16)
layer_sizes = (2, *hidden_sizes, 3)
model_seed = 19

FloatArray = NDArray[np.float64]
IndexArray = NDArray[np.intp]


def make_spiral_dataset(samples_per_class: int, rng: np.random.Generator) -> tuple[FloatArray, IndexArray]:
    classes = np.repeat(np.arange(3), samples_per_class)
    radius = np.tile(np.linspace(0.05, 1.0, samples_per_class), 3)
    angle = classes * (2 * np.pi / 3) + 4 * radius + rng.normal(scale=0.2, size=len(classes))
    inputs = np.column_stack((radius * np.sin(angle), radius * np.cos(angle)))
    inputs += rng.normal(scale=0.04, size=inputs.shape)

    order = rng.permutation(len(classes))
    return inputs[order], classes[order]


train_inputs, train_targets = make_spiral_dataset(5000, rng)
test_inputs, test_targets = make_spiral_dataset(300, rng)


class BatchNormClassifier(Module):
    """An MLP that optionally normalizes each hidden layer before Tanh."""

    def __init__(self, layer_sizes: tuple[int, ...], use_batch_norm: bool, rng: np.random.Generator,
                 ) -> None:
        super().__init__()
        if len(layer_sizes) < 3:
            raise ValueError("BatchNormClassifier needs at least one hidden layer")

        self.hidden_layers: tuple[Linear, ...] = tuple(
            Linear(in_features, out_features, rng)
            for in_features, out_features in zip(layer_sizes, layer_sizes[1:-1])
        )
        self.batch_norm_layers: tuple[BatchNorm1d | None, ...] = tuple(
            BatchNorm1d(layer.out_features) if use_batch_norm else None
            for layer in self.hidden_layers
        )
        self.activation = Tanh()
        self.output_layer = Linear(layer_sizes[-2], layer_sizes[-1], rng)

        self.layers = Sequential(
            *(layer for hidden, batch_norm in zip(self.hidden_layers, self.batch_norm_layers) for layer in
              (hidden, batch_norm, self.activation) if layer is not None),
            self.output_layer,
        )

    def forward_with_final_activation(self, inputs: Leaf | ArrayLike) -> tuple[Leaf, Leaf]:
        values = inputs if isinstance(inputs, Leaf) else Leaf(inputs)
        for hidden, batch_norm in zip(self.hidden_layers, self.batch_norm_layers):
            final_activation_input = hidden(values)
            if batch_norm is not None:
                final_activation_input = batch_norm(final_activation_input)
            values = self.activation(final_activation_input)
        return self.output_layer(values), final_activation_input

    def forward(self, inputs: Leaf | ArrayLike) -> Leaf:
        return self.layers(inputs)

    def parameters(self) -> tuple[Leaf, ...]:
        return self.layers.parameters()

    def train(self) -> "BatchNormClassifier":
        super().train()
        self.layers.train()
        return self

    def eval(self) -> "BatchNormClassifier":
        super().eval()
        self.layers.eval()
        return self


batch_orders = [rng.permutation(len(train_inputs)) for _ in range(epochs)]


def train_epoch(model: BatchNormClassifier, inputs: FloatArray, targets: IndexArray, order: IndexArray) -> float:
    model.train()
    parameters = model.parameters()
    batch_losses: list[float] = []

    for start in range(0, len(order), batch_size):
        indices = order[start:start + batch_size]
        loss = cross_entropy(model(inputs[indices]), targets[indices])
        batch_losses.append(loss.data.item())

        loss.wire()
        for parameter in parameters:
            parameter.data -= learning_rate * parameter.grad
        loss.rest()

    return float(np.mean(batch_losses))


def evaluate(model: BatchNormClassifier, inputs: FloatArray, targets: IndexArray) -> tuple[float, float, FloatArray]:
    model.eval()
    predictions, final_activation_input = model.forward_with_final_activation(inputs)
    loss = cross_entropy(predictions, targets).data.item()
    accuracy = float((predictions.data.argmax(axis=1) == targets).mean())
    return loss, accuracy, final_activation_input.data


def fit(model: BatchNormClassifier) -> tuple[FloatArray, FloatArray, FloatArray, FloatArray]:
    train_losses: list[float] = []
    test_losses: list[float] = []
    test_accuracies: list[float] = []
    final_activation_input: FloatArray | None = None

    for order in batch_orders:
        train_losses.append(train_epoch(model, train_inputs, train_targets, order))
        test_loss, test_accuracy, final_activation_input = evaluate(model, test_inputs, test_targets)
        test_losses.append(test_loss)
        test_accuracies.append(test_accuracy)

    if final_activation_input is None:
        raise RuntimeError("fit needs at least one epoch")
    return np.array(train_losses), np.array(test_losses), np.array(test_accuracies), final_activation_input


cross_entropy = CrossEntropyLoss()
without_batch_norm = BatchNormClassifier(layer_sizes, use_batch_norm=False, rng=np.random.default_rng(model_seed))
with_batch_norm = BatchNormClassifier(layer_sizes, use_batch_norm=True, rng=np.random.default_rng(model_seed))

train_without, test_without, accuracy_without, final_without = fit(without_batch_norm)
train_with, test_with, accuracy_with, final_with = fit(with_batch_norm)

fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
axes[0].plot(train_without, color=COLORS["before"], label="train, without BatchNorm")
axes[0].plot(train_with, color=COLORS["after"], label="train, with BatchNorm")
axes[0].set(title="cross-entropy by epoch", xlabel="epoch", ylabel="loss")
axes[0].legend(fontsize=8)

axes[1].plot(accuracy_without, color=COLORS["before"], label="without BatchNorm")
axes[1].plot(accuracy_with, color=COLORS["after"], label="with BatchNorm")
axes[1].set(title="test accuracy by epoch", xlabel="epoch", ylabel="accuracy", ylim=(0, 1))
axes[1].legend(fontsize=8)

for ax in axes.flat:
    ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()